# Section 1 merge benchmark results

In [1]:
import sys
sys.path.append('./')
from utils import split_paragraphs
from utils import fix_json_string

In [2]:
import ast
import os
import json
import pandas as pd
from re import split
df_all = pd.DataFrame()
folder = "task1_bench_result/"
for file in os.listdir(folder):
    if ".csv" in file:
        try:
            df = pd.read_csv(folder + file)
        except:
            df = pd.read_csv(folder + file, sep = ";")
        print(file)
        for idx, row in df.iterrows():
            response = row["response"].replace("```json", "").replace("```", "").replace("\n[", "[").replace("]\n", "]").replace("*", "")
            response = response.replace('\\n', '\n').replace('\\t', '\t')
            # Apply the fix and parse the JSON
            try:
                fixed_json = fix_json_string(response)
                parsed_data = json.loads(fixed_json)
                #print("Successfully parsed!")
            except Exception as e:
                try:
                    import json5
                    parsed_data = json5.loads(fixed_json)
                    #print("Successfully parsed with json5!")
                except Exception as e2:
                    print(f"json5 error: {e2}")

            d1_ = pd.DataFrame(parsed_data)
            book = row.book
            model = file.split("_")[:-2]
            dd = pd.DataFrame(split_paragraphs(row["pre"]), columns=["no_claims"])
            d1 = pd.concat([d1_, dd.loc[:len(d1_)]], axis = 1)
            d1["book"] = book
            d1["company"] = model[0]
            d1["model"] = model[1]
            df_all = pd.concat([df_all, d1])

qwen_qwen3-30b-a3b-instruct-2507_task1_prompt.csv
openai_gpt-5.1_task1_prompt.csv
openai_gpt-4o-2024-11-20_task1_prompt.csv
qwen_qwen3-30b-a3b-thinking-2507_task1_prompt.csv
openai_gpt-4o-mini-2024-07-18_task1_prompt.csv
mistralai_mistral-small-creative_task1_prompt.csv
x-ai_grok-3_task1_prompt.csv
ai21_jamba-large-1.7_task1_prompt.csv
mistralai_mistral-large-2512_task1_prompt.csv
ai21_jamba-mini-1.7_task1_prompt.csv
meta-llama_llama-3.1-405b-instruct_task1_prompt.csv
google_gemini-2.5-flash-lite_task1_prompt.csv
deepseek_deepseek-v3.2-exp_task1_prompt.csv
deepseek_deepseek-r1-0528_task1_prompt.csv
google_gemini-2.5-flash-preview-09-2025_task1_prompt.csv
anthropic_claude-3.7-sonnet_task1_prompt.csv
meta-llama_llama-3.3-70b-instruct_task1_prompt.csv
x-ai_grok-3-mini_task1_prompt.csv
anthropic_claude-3.7-sonnet:thinking_task1_prompt.csv
mistralai_mistral-small-3.2-24b-instruct_task1_prompt.csv
qwen_qwen3-235b-a22b-2507_task1_prompt.csv
qwen_qwen3-235b-a22b-thinking-2507_task1_prompt.csv


In [3]:
gold = pd.read_csv('task1_dataset/task1_annotation.csv')
print(gold.columns)
gold["pre"] = gold.apply(lambda x: str(int(x["idx_claim"])) + "." + x["Claim"], axis =1)
gold.shape

Index(['book', 'No.orig', 'idx_claim', 'idx_book_claim', 'pub_year', 'context',
       'quote', 'Claim', 'agreed_labels', 'Reasoning_manullyverified',
       'Aspect', 'difficulty_level'],
      dtype='object')


(299, 13)

In [4]:
gold.agreed_labels.describe()
pd.pivot_table(
    data=gold, 
    index='difficulty_level',
    columns='agreed_labels',
    aggfunc='size'
)


agreed_labels,False,True
difficulty_level,,
High,40,13
Low,63,77
Medium,56,50


In [5]:
df_all["idx_claim"] = df_all.no_claims.apply(lambda x: int(x.split(".")[0]))

In [6]:
merged_df = pd.merge(gold, df_all, on = ["book", "idx_claim"], how = "outer") 

In [8]:
#merged_df.to_csv("task1_dataset/task1_claim_result.csv", index = False)

# Section 2 results analysis 

In [1]:
import pandas as pd
import numpy as np
from sklearn.metrics import confusion_matrix

In [2]:
merged_df = pd.read_csv("task1_dataset/task1_claim_result.csv")
merged_df["y_true"] = merged_df.agreed_labels.apply(lambda x: 1 if x else 0)

In [3]:
def map(x):
    if x is not np.nan:
        if (x.lower() == "true") or ("true" in x.lower()):
            return 1
        elif (x.lower() == "false") or ("false" in x.lower()):
            return 0
    else:
        return None

merged_df["pred"] = merged_df.judgment.apply(map)

In [4]:
# Function to calculate metrics from confusion matrix
def get_metrics_from_cm(y_true, y_pred):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0
    f1 = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0
    
    return {
        'true_positive': tp,
        'false_positive': fp,
        'false_negative': fn,
        'true_negative': tn,
        'precision': precision,
        'recall': recall,
        'f1_score': f1
    }

# Group by book_id and calculate metrics
results = []
for meta, group_ in merged_df.groupby(['company', 'model', 'difficulty_level']):
    group = group_.dropna()
    if len(group) != len(group_):
        print("Removoed Nan: ", len(group_)-len(group))
    metrics = get_metrics_from_cm(group['y_true'], group['pred'])
    metrics['model'] = meta[1]
    metrics['company'] = meta[0]
    metrics["difficulty_level"] = meta[2]
    results.append(metrics)

# Convert results to DataFrame
metrics_df = pd.DataFrame(results)
print("Metrics by model using Confusion Matrix:")


Removoed Nan:  1
Removoed Nan:  1
Removoed Nan:  1
Removoed Nan:  1
Removoed Nan:  1
Removoed Nan:  1
Removoed Nan:  1
Removoed Nan:  1
Removoed Nan:  1
Removoed Nan:  1
Removoed Nan:  1
Removoed Nan:  1
Removoed Nan:  1
Removoed Nan:  1
Removoed Nan:  1
Removoed Nan:  1
Removoed Nan:  1
Removoed Nan:  1
Removoed Nan:  1
Removoed Nan:  1
Removoed Nan:  3
Removoed Nan:  1
Removoed Nan:  3
Removoed Nan:  1
Removoed Nan:  1
Removoed Nan:  1
Removoed Nan:  1
Metrics by model using Confusion Matrix:


In [5]:

metrics_df['difficulty_level'] = pd.Categorical(
    metrics_df['difficulty_level'],
    categories=['Low', 'Medium', 'High'],
    ordered=True
)
metrics_df.sort_values(["company", "model", "difficulty_level", "f1_score"]).to_csv("result/task1_analysis_per_diff.csv")

In [6]:
# Group by book_id and calculate metrics
results = []
for meta, group_ in merged_df.groupby(['company', 'model']):
    group = group_.dropna()
    if len(group) != len(group_):
        print("Removoed Nan: ", len(group_)-len(group))
    metrics = get_metrics_from_cm(group['y_true'], group['pred'])
    metrics['model'] = meta[1]
    metrics['company'] = meta[0]
    results.append(metrics)

# Convert results to DataFrame
metrics_df_ = pd.DataFrame(results)
print("Metrics by model using Confusion Matrix:")
metrics_df_.sort_values(["company", "model", "f1_score"]).to_csv("result/task1_analysis_all.csv")


Removoed Nan:  1
Removoed Nan:  3
Removoed Nan:  1
Removoed Nan:  1
Removoed Nan:  1
Removoed Nan:  1
Removoed Nan:  1
Removoed Nan:  1
Removoed Nan:  1
Removoed Nan:  2
Removoed Nan:  1
Removoed Nan:  1
Removoed Nan:  1
Removoed Nan:  1
Removoed Nan:  1
Removoed Nan:  1
Removoed Nan:  4
Removoed Nan:  1
Removoed Nan:  3
Removoed Nan:  1
Removoed Nan:  1
Removoed Nan:  1
Removoed Nan:  1
Metrics by model using Confusion Matrix:


In [7]:
merged_df.book.unique()
trans_anno = ['David_copperfield', 'uncle_toms_cabin',
    'frankenstain', 'moby_dick',
       'gone_with_the_wind', 'jane_eyre',
        'mrs.dalloway', 
        'wuthering_height']

In [8]:
# 8 books
results = []
sub_bookdf = merged_df.set_index("book").loc[trans_anno].reset_index()
for meta, group_ in sub_bookdf.groupby(['company', 'model']):
    group = group_.dropna()
    if len(group) != len(group_):
        print("Removoed Nan: ", len(group_)-len(group))
    metrics = get_metrics_from_cm(group['y_true'], group['pred'])
    metrics['model'] = meta[1]
    metrics['company'] = meta[0]
    results.append(metrics)

# Convert results to DataFrame
metrics_df__ = pd.DataFrame(results)
print("Metrics by model using Confusion Matrix:")
metrics_df__.sort_values(["company", "model", "f1_score"]).to_csv("result/task1_analysis_sub8.csv")


Removoed Nan:  1
Removoed Nan:  3
Removoed Nan:  1
Removoed Nan:  1
Removoed Nan:  1
Removoed Nan:  1
Removoed Nan:  1
Removoed Nan:  1
Removoed Nan:  1
Removoed Nan:  2
Removoed Nan:  1
Removoed Nan:  1
Removoed Nan:  1
Removoed Nan:  1
Removoed Nan:  1
Removoed Nan:  1
Removoed Nan:  3
Removoed Nan:  1
Removoed Nan:  1
Removoed Nan:  1
Removoed Nan:  1
Removoed Nan:  1
Removoed Nan:  1
Metrics by model using Confusion Matrix:


In [ ]:
print(metrics_df__)

,true_positive,false_positive,false_negative,true_negative,precision,recall,f1_score,model,company
0,92,18,1,106,0.836364,0.989247,0.906404,jamba-large-1.7,ai21
1,87,25,5,98,0.776786,0.945652,0.852941,jamba-mini-1.7,ai21
2,88,12,5,112,0.880000,0.946237,0.911917,claude-3.7-sonnet,anthropic
3,84,7,9,117,0.923077,0.903226,0.913043,claude-3.7-sonnet:thinking,anthropic
4,83,15,10,109,0.846939,0.892473,0.869110,deepseek-r1-0528,deepseek
5,84,12,9,112,0.875000,0.903226,0.888889,deepseek-v3.2-exp,deepseek
6,82,10,11,114,0.891304,0.881720,0.886486,gemini-2.5-flash-lite,google
7,86,8,7,116,0.914894,0.924731,0.919786,gemini-2.5-flash-preview-09-2025,google
8,88,11,5,113,0.888889,0.946237,0.916667,llama-3.1-405b-instruct,meta-llama
9,87,17,5,107,0.836538,0.945652,0.887755,llama-3.3-70b-instruct,meta-llama
